# 02 — Indexing Pipeline

End-to-end run of the full indexing pipeline on a sample:
1. Load → Clean → Chunk
2. Build Chroma vector store
3. Build BM25 index
4. Build relationship graph
5. Verify all indexes with test queries

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

from configs import config

## 1. Load and process a small sample

In [ ]:
from src.ingestion.loader import load_documents
from src.ingestion.cleaner import clean_documents
from src.ingestion.chunker import chunk_documents

SAMPLE_SIZE = 500

print(f"Loading {SAMPLE_SIZE} documents...")
docs = load_documents(config, sample_size=SAMPLE_SIZE)
print(f"Cleaning...")
cleaned = clean_documents(docs)
print(f"Chunking...")
chunks = chunk_documents(cleaned, config)
print(f"✓ {len(docs)} docs → {len(chunks)} chunks")

## 2. Build Chroma vector store

In [ ]:
from src.indexing.chroma_store import build_store_from_chunks

print("Building Chroma store (embedding + upsert, may take a few minutes)...")
store = build_store_from_chunks(chunks)
count = store._collection.count()
print(f"✓ Chroma: {count} vectors indexed @ {config.chroma.host}:{config.chroma.port}")

## 3. Build BM25 index

In [ ]:
from src.indexing.bm25_index import build_bm25_index

bm25 = build_bm25_index(chunks, save_path='../data/processed/bm25_sample.pkl')
print(f"✓ BM25 index built over {len(chunks)} chunks")

## 4. Build relationship graph

## 5. Verify all indexes with test queries

from src.retrieval.dense import dense_search
from src.retrieval.hybrid import hybrid_search
from src.retrieval.reranker import rerank
from src.retrieval.graph import graph_search

TEST_QUERY = "Luật đất đai có hiệu lực khi nào?"

print("=== Dense Search ===")
dense_docs = dense_search(store, TEST_QUERY, k=3)
for d in dense_docs:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")

print("\n=== Hybrid Search ===")
hybrid_docs = hybrid_search(store, bm25, TEST_QUERY, k=3)
for d in hybrid_docs:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")

print("\n=== Reranked ===")
candidates = hybrid_search(store, bm25, TEST_QUERY, k=6)
reranked = rerank(TEST_QUERY, candidates, k=3)
for d in reranked:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")

print("\n=== Graph Search ===")
graph_docs = graph_search(store, G, TEST_QUERY, k=3)
for d in graph_docs:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")

In [ ]:
from src.retrieval.dense import dense_search
from src.retrieval.hybrid import hybrid_search
from src.retrieval.reranker import rerank

TEST_QUERY = "Luật đất đai có hiệu lực khi nào?"

print("=== Dense Search ===")
dense_docs = dense_search(store, TEST_QUERY, k=3)
for d in dense_docs:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")

print("\n=== Hybrid Search ===")
hybrid_docs = hybrid_search(store, bm25, TEST_QUERY, k=3)
for d in hybrid_docs:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")

print("\n=== Reranked ===")
candidates = hybrid_search(store, bm25, TEST_QUERY, k=6)
reranked = rerank(TEST_QUERY, candidates, k=3)
for d in reranked:
    print(f"  [{d.metadata.get('doc_id','')}] {d.page_content[:150]}")